In [75]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

Libraries imported successfully!


In [76]:
data_path = "data/wdbc.data"

df = pd.read_csv(data_path, header=None)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (569, 32)


In [77]:
columns = [
    "id",
    "diagnosis",

    "radius_mean",
    "texture_mean",
    "perimeter_mean",
    "area_mean",
    "smoothness_mean",
    "compactness_mean",
    "concavity_mean",
    "concave_points_mean",
    "symmetry_mean",
    "fractal_dimension_mean",

    "radius_se",
    "texture_se",
    "perimeter_se",
    "area_se",
    "smoothness_se",
    "compactness_se",
    "concavity_se",
    "concave_points_se",
    "symmetry_se",
    "fractal_dimension_se",

    "radius_worst",
    "texture_worst",
    "perimeter_worst",
    "area_worst",
    "smoothness_worst",
    "compactness_worst",
    "concavity_worst",
    "concave_points_worst",
    "symmetry_worst",
    "fractal_dimension_worst"
]

df.columns = columns

print("Column names assigned successfully!")
print(df.shape)

Column names assigned successfully!
(569, 32)


In [78]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [79]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave_points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             569 non-null

In [80]:
df["diagnosis"].value_counts()

diagnosis
B    357
M    212
Name: count, dtype: int64

In [81]:
# Remove the ID column
df = df.drop("id", axis=1)

# Separate features (X) and target (y)
X = df.drop("diagnosis", axis=1)
y = df["diagnosis"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget values:")
print(y.value_counts())

Features shape: (569, 30)
Target shape: (569,)

Target values:
diagnosis
B    357
M    212
Name: count, dtype: int64


In [82]:
y = y.map({"B": 0, "M": 1})

print(y.value_counts())

diagnosis
0    357
1    212
Name: count, dtype: int64


In [83]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (455, 30)
Testing data: (114, 30)


In [84]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed successfully!")

Scaling completed successfully!


## Model selection and final evaluation

Candidate K values are compared with 5-fold cross-validation using training data only. K is selected by highest mean CV accuracy; ties are broken by lower CV standard deviation, then lower K. The held-out test set is reserved for one final evaluation after selection; it is not used to choose K.

In [11]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

# Select K using 5-fold cross-validation on training data only.
# X_test_scaled and y_test are not used during model selection.
k_range = range(1, 21)
cv_results = {}
for k in k_range:
    cv_results[k] = cross_val_score(
        KNeighborsClassifier(n_neighbors=k),
        X_train_scaled,
        y_train,
        cv=5,
        scoring="accuracy"
    )

cv_means = {k: scores.mean() for k, scores in cv_results.items()}
cv_stds = {k: scores.std() for k, scores in cv_results.items()}
top_mean = max(cv_means.values())
top_candidates = [k for k in k_range if np.isclose(cv_means[k], top_mean)]
# Tie-break equal mean scores by lower fold standard deviation, then lower K.
best_k = min(top_candidates, key=lambda k: (cv_stds[k], k))
best_cv_scores = cv_results[best_k]
best_cv_mean = best_cv_scores.mean()
best_cv_std = best_cv_scores.std()

for k in k_range:
    scores = cv_results[k]
    print(f"K={k:2d}: mean accuracy={scores.mean() * 100:.2f}%, std={scores.std() * 100:.2f}%")
print(f"Top mean CV candidates: {top_candidates}")
print(f"Selected K={best_k} using training-only 5-fold CV (lowest CV standard deviation among top means).")
print(f"Selected K mean accuracy: {best_cv_mean * 100:.2f}%")
print(f"Selected K standard deviation: {best_cv_std * 100:.2f}%")
assert best_k == 3, f"Expected K=3 from the recorded CV selection, got K={best_k}."


K= 1: mean accuracy=94.73%, std=1.46%
K= 2: mean accuracy=95.60%, std=1.97%
K= 3: mean accuracy=96.92%, std=2.13%
K= 4: mean accuracy=96.04%, std=2.66%
K= 5: mean accuracy=96.26%, std=2.37%
K= 6: mean accuracy=95.82%, std=2.13%
K= 7: mean accuracy=96.92%, std=2.35%
K= 8: mean accuracy=96.26%, std=1.92%
K= 9: mean accuracy=96.48%, std=1.89%
K=10: mean accuracy=96.04%, std=1.79%
K=11: mean accuracy=95.82%, std=2.13%
K=12: mean accuracy=95.82%, std=1.89%
K=13: mean accuracy=95.82%, std=1.89%
K=14: mean accuracy=95.60%, std=1.97%
K=15: mean accuracy=96.04%, std=1.79%
K=16: mean accuracy=95.82%, std=1.89%
K=17: mean accuracy=96.04%, std=1.79%
K=18: mean accuracy=95.60%, std=1.84%
K=19: mean accuracy=95.60%, std=1.84%
K=20: mean accuracy=95.60%, std=1.84%
Top mean CV candidates: [3, 7]
Selected K=3 using training-only 5-fold CV (lowest CV standard deviation among top means).
Selected K mean accuracy: 96.92%
Selected K standard deviation: 2.13%


In [12]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier

# Fit the selected final model on all training rows.
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train_scaled, y_train)

# Use the held-out test set only for this final evaluation.
final_predictions = final_knn.predict(X_test_scaled)
final_test_accuracy = accuracy_score(y_test, final_predictions)
final_classification_report = classification_report(
    y_test, final_predictions, target_names=["Benign", "Malignant"]
)
final_confusion_matrix = confusion_matrix(y_test, final_predictions)

print(f"Final K: {final_knn.n_neighbors}")
print(f"Final held-out test accuracy: {final_test_accuracy * 100:.2f}%")
print("Classification report:")
print(final_classification_report)
print("Confusion matrix (actual rows, predicted columns; Benign, Malignant):")
print(final_confusion_matrix)


Final K: 3
Final held-out test accuracy: 93.86%
Classification report:
              precision    recall  f1-score   support

      Benign       0.92      0.99      0.95        72
   Malignant       0.97      0.86      0.91        42

    accuracy                           0.94       114
   macro avg       0.95      0.92      0.93       114
weighted avg       0.94      0.94      0.94       114

Confusion matrix (actual rows, predicted columns; Benign, Malignant):
[[71  1]
 [ 6 36]]


In [13]:
import joblib

# Save only the final K=3 classifier. Keep the existing scaler unchanged.
assert final_knn.n_neighbors == 3
joblib.dump(final_knn, "backend/model/knn_model.pkl")
print("Saved final K=3 model to backend/model/knn_model.pkl")


Saved final K=3 model to backend/model/knn_model.pkl
